In [44]:
import os, sys, re
from typing import List, Dict, Optional

project_root = os.path.dirname(os.getcwd())
# print(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd

import traceback
import importlib, src.data_ingestion as _di
importlib.reload(_di)

from src.data_ingestion import load_data_from_gcs, SERVICE_ACCOUNT_PATH, bucket_name, blob_name, load_processed_data_from_gcs




In [45]:
# exp_text = "i ve learned this lesson again open the package and use the product right away ordered this mouse in august as my travel mouse and just packed it away in my bag now ve been visiting family in the pnw since mid september the mouse took charge and worked fine for couple of weeks after recharged the mouse it worked for day or two and then ceased to function the optical light will flash once when turned on then nothing of course the return window is well past "

def tokenize(text: str) -> list:
    text = text.lower()
    text = re.sub(pattern=r'[^a-z0-9]\s', repl=" ", string=text)
    return text.split()



In [46]:
def join_tokens(tokens: list) -> str:
    return " ".join(tokens)


In [47]:
join_tokens(["I", "live", "in", "hyderabad"])

'I live in hyderabad'

In [48]:
def preprocess_series(series: pd.Series) -> pd.Series:
    return series.dropna().apply(lambda t: join_tokens(tokenize(t)))



In [49]:
def get_vocab_stats(corpus: pd.Series) -> dict:
    all_tokens = corpus.dropna().apply(tokenize)
    vocab = {token for tokens in all_tokens for token in tokens}
    avg_len = all_tokens.apply(len).mean()

    return {
        "vocab_size": len(vocab),
        "avg_tokens_per_doc": round(avg_len, 2),
        "total_doc": len(corpus.dropna()), 
    }

In [50]:
# load the data from gcs

df = load_data_from_gcs("cleaned_reviews.csv")
print(df.shape)

Downloaded file to: C:\Users\PRIYAB~1\AppData\Local\Temp\cleaned_reviews.csv
(17340, 4)


In [51]:
df.isnull().sum()

sentiments               0
cleaned_review           3
cleaned_review_length    0
review_score             0
dtype: int64

In [52]:
samples = df['cleaned_review'].dropna().sample(4, random_state=42).tolist()

In [53]:
samples

['it meets my need ',
 'i shopped around for while great speaker sound excellent very happy',
 'why there no lights or showing if it charged or not have to turn it on to see the lights to see if it charge do you have to do that all the time turn it on to see if it charged or can you just plug it in and the lights will show if it when it going to be fully chargeyou have to turn it on to see the lights to see if it charge do you have to do that all the time turn it on to see if it charged or can you just plug it in and the lights will show maybe something wrong with mine please let me know',
 'it had connectivity issues right out of the box contacted the seller and he she did not respond it never worked even if it had it is so misshapen it isn comfortable to use cannot recommend seller or product ']

In [54]:
for i, raw_text in enumerate(samples, 1):
    tokens = tokenize(raw_text)
    print(f"Example {i}")
    print(f" Input: {raw_text[:90]}")
    print(f"Tokens: {tokens[:10]}")
    print(f" Count: {len(tokens)}")

Example 1
 Input: it meets my need 
Tokens: ['it', 'meets', 'my', 'need']
 Count: 4
Example 2
 Input: i shopped around for while great speaker sound excellent very happy
Tokens: ['i', 'shopped', 'around', 'for', 'while', 'great', 'speaker', 'sound', 'excellent', 'very']
 Count: 11
Example 3
 Input: why there no lights or showing if it charged or not have to turn it on to see the lights t
Tokens: ['why', 'there', 'no', 'lights', 'or', 'showing', 'if', 'it', 'charged', 'or']
 Count: 115
Example 4
 Input: it had connectivity issues right out of the box contacted the seller and he she did not re
Tokens: ['it', 'had', 'connectivity', 'issues', 'right', 'out', 'of', 'the', 'box', 'contacted']
 Count: 39


In [55]:
stats = get_vocab_stats(df['cleaned_review'])
print("Corpus vocabulary statistics")
for k, v in stats.items():
    print(f"{k:30s}: {v}")

Corpus vocabulary statistics
vocab_size                    : 9593
avg_tokens_per_doc            : 30.31
total_doc                     : 17337


In [56]:
samples_idx = df['cleaned_review'].dropna().sample(3, random_state=7).index
print("Before and After preprocessing...")
print("-" * 100)

for idx in samples_idx:
    original = df.loc[idx, "cleaned_review"]
    processed = join_tokens(tokenize(original))
    print(f"Before: {original[:100]}")
    print(f"After: {processed[:100]}")

    


Before and After preprocessing...
----------------------------------------------------------------------------------------------------
Before: i love this mouse the colors are beautiful and ve gotten many compliments on it it feels great to us
After: i love this mouse the colors are beautiful and ve gotten many compliments on it it feels great to us
Before: at first was little bit worried about the quality of this product but let me tell you this worked ou
After: at first was little bit worried about the quality of this product but let me tell you this worked ou
Before: these headphones are amazing especially for the price the cable is long and the braids make it tough
After: these headphones are amazing especially for the price the cable is long and the braids make it tough


In [57]:
cleaned_series = preprocess_series(df["cleaned_review"])

print(f"Rows before preprocessing: {len(df)}")
print(f"Rows after preprocessing: {len(cleaned_series)}")

print(cleaned_series.head(5).to_string())


Rows before preprocessing: 17340
Rows after preprocessing: 17337
0    i wish would have gotten one earlier love it a...
1    i ve learned this lesson again open the packag...
2            it is so slow and lags find better option
3    roller ball stopped working within months of m...
4    i like the color and size but it few days out ...


In [58]:
cleaned_series

0        i wish would have gotten one earlier love it a...
1        i ve learned this lesson again open the packag...
2                it is so slow and lags find better option
3        roller ball stopped working within months of m...
4        i like the color and size but it few days out ...
                               ...                        
17335    i love this speaker and love can take it anywh...
17336    i use it in my house easy to connect and loud ...
17337    the bass is good and the battery is amazing mu...
17338                                              love it
17339                                         mono speaker
Name: cleaned_review, Length: 17337, dtype: str

In [59]:
df_processed = df.loc[cleaned_series.index].copy()
df_processed['cleaned_review'] = cleaned_series
df_processed['word_count'] = cleaned_series.str.split().str.len()

df_processed = df_processed[["sentiments", "cleaned_review", "review_score", "word_count"]]
df_processed = df_processed.reset_index(drop=True)

print(f"Processed dataset shape: {df_processed.shape}")
print(df_processed['sentiments'].value_counts())
print(df_processed.head())



Processed dataset shape: (17337, 4)
sentiments
positive    9503
neutral     6300
negative    1534
Name: count, dtype: int64
  sentiments                                     cleaned_review  review_score  \
0   positive  i wish would have gotten one earlier love it a...           5.0   
1    neutral  i ve learned this lesson again open the packag...           1.0   
2    neutral          it is so slow and lags find better option           2.0   
3    neutral  roller ball stopped working within months of m...           1.0   
4    neutral  i like the color and size but it few days out ...           1.0   

   word_count  
0          19  
1          88  
2           9  
3          12  
4          21  


In [61]:


df_verify = load_processed_data_from_gcs("processed_reviews.csv")

print(f"Reloaded from GCS: {df_verify.shape}")
print(df_verify.head().to_string())


Reloaded from GCS: (17321, 4)
  sentiments                                                                                                                                                                                                                                                                                                                                                                                                                                                               cleaned_review  review_score  word_count
0   positive                                                                                                                                                                                                                                                                                                                                                                                i wish would have gotten one earlier love it and it makes working in my laptop so much eas